In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


#1. 入力状態
class InputState(TypedDict):
    username: str


#2. 出力状態
class OutputState(TypedDict):
    graph_output: str


#3. グローバル状態
class OverAllState(TypedDict):
    username: str
    graph_output: str
    nickname: str


#4. プライベート状態
class PrivateState(TypedDict):
    greeting: str


#5. 1つ目のノード。start に接続 => InputState。変更した状態の内容はグローバル状態に含まれる -> OverAllState
def node_1(state: InputState) -> OverAllState:
    # グローバル状態に username を追加
    return {
        "nickname": "Dear " + state["username"]
    }


#6. 2つ目のノード。node1 に接続 => OverAllState。使用するパラメータはグローバル状態にあり、変更するパラメータはプライベート状態にある -> PrivateState
def node_2(state: OverAllState) -> PrivateState:
    # プライベート状態に greeting を追加
    return {
        "greeting": "Hello, " + state["nickname"]
    }


#7. 3つ目のノード。node2 に接続 => PrivateState。使用するパラメータはプライベート状態にあり、変更するパラメータは出力状態にある -> OutputState
def node_3(state: PrivateState) -> OutputState:
    # 出力状態に graph_output を追加
    return {
        "graph_output": state["greeting"] + " お会いできて嬉しいです! "
    }


#8. 状態グラフを構築
# グラフを定義する際に、グローバル状態・入力状態・出力状態を読み込む
builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState)

#9. ノードを追加
# ノードを追加する際に、プライベート状態を読み込む
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

#10. エッジを追加
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
# 入力する状態
result = graph.invoke({"username": "atguigu"})
# 結果を出力 => 出力状態
print(result)


In [ ]:
from IPython.display import display

display(graph)